In [ ]:
import pandas as pd
import geopandas as gpd
from shapely import Point

In [ ]:
# Load birds data into a pandas dataframe
input_csv = pd.read_csv('../data/black-tailed.csv', index_col=0)

# Get list of birds names
NAMES = sorted(list(set(input_csv['individual-local-identifier'])))
print(NAMES)

# Set a filter to select only the birds in Rotterdam
input_csv = input_csv[input_csv['individual-local-identifier'] == 'Rotterdam']

# Remove rows that have no coordinates
data = input_csv[input_csv['location-long'].notnull()]

In [ ]:
# Create a geometry column
birds_locations = data.apply(lambda x: Point(x['location-long'], x['location-lat']), axis=1)

# Convert to a GeoDataFrame
gdf = gpd.GeoDataFrame(data, geometry=birds_locations)
gdf = gdf.set_crs('epsg:4326')

# Project to Web Mercator
gdf = gdf.to_crs('epsg:3857')

In [ ]:
COORDS = []
for i in range(len(gdf)):
    c = [list(gdf.geometry.x)[i], list(gdf.geometry.y)[i]]
    COORDS.append(c)

In [ ]:
# DBSCAN clustering on the birds locations
from sklearn.cluster import DBSCAN

# Compute DBSCAN
clustering = DBSCAN(eps=15000, min_samples=25).fit(COORDS)
labels = clustering.labels_
print(len(set(labels)))

In [ ]:
# Plot the data
gdf.explore()